# 03-9. モンテカルロと MCMC — 動かして確かめる

📖 解説: [`../09_mcmc.md`](../09_mcmc.md)

上から順に **Shift+Enter** で実行していってください。

## このノートで触るもの
1. 積分を標本平均に置き換える — $\mathbb{E}[g(\theta)\mid x] \approx \frac{1}{K}\sum_k g(\theta^{(k)})$
2. Metropolis 法を素朴に実装する（答えの分かる問題で検算）
3. 【対話】`proposal_sd` を壊してみる — 何がどう崩れるか
4. 収束診断 — トレース・受容率・複数連鎖・$\hat{R}$
5. JAX 版 (`jit` + `lax.scan` + `vmap`) と速度比較
6. 【対話】階層モデルと縮小 — $n$ の小さい店舗を守る
7. 事後予測チェック — 推定できてもモデルが正しいとは限らない
8. 逐次更新 — 昨日の事後が今日の事前になる

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (09_mcmc.md)](../09_mcmc.md)

In [ ]:
import time

import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", message=".*distutils Version classes.*", category=DeprecationWarning)
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)
from ipywidgets import interact, IntSlider, FloatSlider

%matplotlib inline

rng = np.random.default_rng(42)

# この章を通して使うモデル: 事前 Beta(2,2) × 「20人中12人成約」-> 事後 Beta(14,10)
A0: float = 2.0      # 事前 Beta(α₀, β₀)
B0: float = 2.0
N: int = 20          # 試行数 (人)
Y: int = 12          # 成功数 (人)

exact = stats.beta(A0 + Y, B0 + N - Y)   # 厳密な事後分布 Beta(14, 10)
print(f'厳密な事後分布: Beta({A0+Y:.0f}, {B0+N-Y:.0f}),  事後平均 = {exact.mean():.4f}')

## 1. 積分を標本平均に置き換える

欲しいものは、ほとんど全部が積分の形をしています。

$$
\mathbb{E}[g(\theta) \mid x] = \int g(\theta)\pi(\theta \mid x)\,d\theta,
\qquad
P(\theta > c \mid x) = \int I(\theta > c)\pi(\theta \mid x)\,d\theta
$$

**標本さえあれば**、これらは全部「並べて数える」だけで求まります。

$$
\mathbb{E}[g(\theta) \mid x] \approx \frac{1}{K}\sum_{k=1}^{K} g(\theta^{(k)}),
\qquad
P(\theta > c \mid x) \approx \frac{1}{K}\sum_{k=1}^{K} I(\theta^{(k)} > c)
$$

まずは（今回は名前の付く分布なので）**直接引ける標本**で、この近似が効くことを確認します。

In [ ]:
THRESHOLD: float = 0.50     # しきい値 c
K_LIST = [10, 100, 1_000, 10_000, 100_000]

print(f'{"K":>8} {"標本平均":>10} {"P(θ>0.5)":>10}   (厳密: '
      f'{exact.mean():.4f} / {1 - exact.cdf(THRESHOLD):.4f})')
print('-' * 60)
means, probs = [], []
for K in K_LIST:
    draws = exact.rvs(K, random_state=np.random.default_rng(0))   # shape: (K,)
    m = float(draws.mean())                                        # g(θ) = θ
    p = float((draws > THRESHOLD).mean())                          # g(θ) = I(θ > c)
    means.append(m); probs.append(p)
    print(f'{K:>8} {m:>10.4f} {p:>10.4f}')

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, vals, exact_v, title in [
    (axes[0], means, exact.mean(), '事後平均 E[θ|x]'),
    (axes[1], probs, 1 - exact.cdf(THRESHOLD), 'P(θ > 0.5 | x)'),
]:
    ax.semilogx(K_LIST, vals, 'o-', lw=2)
    ax.axhline(exact_v, color='C3', ls='--', label=f'厳密解 {exact_v:.4f}')
    ax.set_xlabel('標本数 K'); ax.set_title(title); ax.legend()
plt.suptitle('標本を増やすほど、標本平均は積分の値に近づく (大数の法則)')
plt.tight_layout(); plt.show()

### 読み取り

$K$ を増やすほど厳密解に近づきます。根拠は[大数の法則](../01_probability_basics.md)。

**問題は「事後分布から標本をどう取るか」**です。
今回は $\mathrm{Beta}(14,10)$ と名前が付いたので `rvs()` で引けましたが、
非共役モデルではそれができません。そこで MCMC が要ります。

## 2. Metropolis 法 — 一番シンプルな MCMC

1. **提案**: $\theta^{*} \sim N(\theta^{(k)}, s^2)$
2. **比を計算**: $\log r = \log\pi(\theta^{*}\mid x) - \log\pi(\theta^{(k)}\mid x)$
3. **採否**: $\log u < \log r$ なら移動、さもなくば**現在値をもう一度記録**

**正規化定数は要りません**（比を取ると割り算で消える）。だから `log_posterior` は
$\propto$ の分子だけ書けば十分です。

In [ ]:
# --- アルゴリズムのパラメータ (モデルではない: 事後分布そのものは変えない) ---
N_ITER: int = 11_000       # 連鎖の総ステップ数
BURN_IN: int = 1_000       # 捨てる先頭ステップ数
PROPOSAL_SD: float = 0.08  # 一歩の大きさ s
INIT: float = 0.5          # 初期値 (成約率なので中央)


def log_posterior(theta: float) -> float:
    """正規化定数を除いた対数事後密度 log π(θ|x) + const。

    Args:
        theta: 成約率 (0 < theta < 1)。

    Returns:
        対数事後密度。定義域の外は -inf (= 絶対に採択されない)。
    """
    if theta <= 0.0 or theta >= 1.0:
        return -np.inf
    return (A0 + Y - 1) * np.log(theta) + (B0 + N - Y - 1) * np.log(1 - theta)


def metropolis(seed: int, n_iter: int = N_ITER, proposal_sd: float = PROPOSAL_SD,
               init: float = INIT) -> tuple[np.ndarray, float]:
    """Metropolis 法で対数事後密度からサンプリングする。

    Args:
        seed: 乱数シード。
        n_iter: 連鎖の総ステップ数。
        proposal_sd: 提案分布 N(current, proposal_sd²) の標準偏差。
        init: 連鎖の初期値。

    Returns:
        (標本 shape:(n_iter,), 受容率)。
    """
    local_rng = np.random.default_rng(seed)
    samples = np.empty(n_iter)
    samples[0] = init
    current_lp: float = log_posterior(init)
    n_accept: int = 0

    for s in range(1, n_iter):
        current = samples[s - 1]
        proposal = local_rng.normal(current, proposal_sd)     # 現在地の近くに候補
        proposal_lp = log_posterior(proposal)

        # 対数で比較する (確率の積はすぐアンダーフローするため)
        if np.log(local_rng.uniform()) < proposal_lp - current_lp:
            samples[s] = proposal
            current_lp = proposal_lp
            n_accept += 1
        else:
            samples[s] = current      # ⚠️ 棄却されたら現在値をもう一度記録 (仕様)

    return samples, n_accept / (n_iter - 1)


samples_all, accept_rate = metropolis(seed=42)
samples = samples_all[BURN_IN:]        # shape: (10000,) バーンインを捨てる

print(f'受容率            : {accept_rate:.3f}')
print(f'MCMC 事後平均     : {samples.mean():.4f}   厳密: {exact.mean():.4f}')
print(f'MCMC 95% 信用区間 : {np.percentile(samples, [2.5, 97.5]).round(3)}'
      f'   厳密: {exact.ppf([0.025, 0.975]).round(3)}')
print(f'MCMC P(θ>0.5|x)   : {(samples > 0.5).mean():.3f}'
      f'   厳密: {1 - exact.cdf(0.5):.3f}')

In [ ]:
# トレースプロット + ヒストグラム (厳密解と重ねる)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={'width_ratios': [2, 1.4]})

axes[0].plot(samples_all[:2000], lw=0.6)
axes[0].axvspan(0, BURN_IN, color='C3', alpha=0.15, label=f'バーンイン ({BURN_IN} 回)')
axes[0].set_xlabel('反復回数'); axes[0].set_ylabel('θ')
axes[0].set_title('トレースプロット (先頭 2000 ステップ)')
axes[0].legend()

th = np.linspace(0.001, 0.999, 500)
axes[1].hist(samples, bins=40, density=True, alpha=0.45, label='MCMC 標本')
axes[1].plot(th, stats.beta.pdf(th, A0, B0), lw=2, label=f'事前 Beta({A0:.0f},{B0:.0f})')
axes[1].plot(th, exact.pdf(th), lw=2.5, ls='--', label=f'厳密解 Beta({A0+Y:.0f},{B0+N-Y:.0f})')
axes[1].set_xlabel('成約率 θ'); axes[1].set_ylabel('密度')
axes[1].set_title('ヒストグラムが厳密解に重なるか')
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

# ⚠️ 「同じ値が連続して並ぶ」のはバグではなく仕様 (滞在時間が確率になる)
n_stuck = int((np.diff(samples_all) == 0).sum())
print(f'値が動かなかったステップ数: {n_stuck} / {N_ITER - 1}  ← 棄却された回数。仕様です')

## 3. 【対話】`proposal_sd` を壊してみる

**一歩の大きさ**を極端にすると何が起きるか。

- **極端に小さい** (0.001): ちょろちょろしか動かず、分布全体を覆えない（受容率は高いのに動かない）
- **極端に大きい** (0.5): 候補が端に飛んで**棄却だらけ**（受容率が激減）

> 📌 これらは**モデルではなくアルゴリズムのパラメータ**です。事後分布そのものは変わりません。
> 結果がズレて見えたら「モデルが変わった」のではなく「**収束していない**」という診断情報。

In [ ]:
def explore_proposal(proposal_sd: float = 0.08, burn_in: int = 1000, n_iter: int = 11000) -> None:
    """proposal_sd を変えて、トレースと事後分布の再現度を確認する。

    Args:
        proposal_sd: 提案分布の標準偏差 (一歩の大きさ)。
        burn_in: 捨てる先頭ステップ数。
        n_iter: 連鎖の総ステップ数。
    """
    burn_in = min(burn_in, n_iter - 100)
    chain, acc = metropolis(seed=42, n_iter=n_iter, proposal_sd=proposal_sd)
    kept = chain[burn_in:]

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6), gridspec_kw={'width_ratios': [2, 1.4]})
    axes[0].plot(chain[:min(3000, n_iter)], lw=0.6)
    axes[0].set_ylim(0, 1); axes[0].set_xlabel('反復回数'); axes[0].set_ylabel('θ')
    axes[0].set_title(f'トレース (proposal_sd={proposal_sd}, 受容率={acc:.3f})')

    th = np.linspace(0.001, 0.999, 400)
    axes[1].hist(kept, bins=40, density=True, alpha=0.45, label='MCMC')
    axes[1].plot(th, exact.pdf(th), lw=2.5, ls='--', color='C3', label='厳密解')
    axes[1].set_xlim(0, 1); axes[1].legend(fontsize=9); axes[1].set_xlabel('θ')
    plt.tight_layout(); plt.show()

    err = abs(float(kept.mean()) - float(exact.mean()))
    verdict = '✅ 一致' if err < 0.01 else '⚠️ ズレている (収束していない可能性)'
    print(f'MCMC 事後平均 = {kept.mean():.4f}  厳密 = {exact.mean():.4f}  誤差 = {err:.4f}  {verdict}')


interact(explore_proposal,
         proposal_sd=FloatSlider(value=0.08, min=0.001, max=0.6, step=0.001,
                                 readout_format='.3f', description='一歩の大きさ'),
         burn_in=IntSlider(value=1000, min=0, max=5000, step=100, description='バーンイン'),
         n_iter=IntSlider(value=11000, min=1000, max=21000, step=1000, description='反復回数'));

## 4. 収束診断 — 「十分長く回した」をどう確かめるか

MCMC は十分長く回せば正しい分布に収束しますが、**十分かどうかは自動では分かりません**。

| 診断 | 危険信号 |
|---|---|
| トレースプロット | 一方向にドリフトする／太い帯にならない |
| 受容率 | 極端に低い (<10%) ／極端に高い (>95%)。1 次元なら 0.3〜0.5 が目安 |
| 複数連鎖 | 連鎖どうしが**別の場所に居座る** |
| $\hat{R}$ (R ハット) | 1.01 より大きい |

**別々の初期値**から 4 本走らせて重ねてみます。

In [ ]:
def gelman_rubin(chains: np.ndarray) -> float:
    """Gelman-Rubin の R̂ (簡易版) を計算する。

    連鎖間のばらつきと連鎖内のばらつきを比べ、1 に近ければ収束とみなす。

    Args:
        chains: shape (n_chains, n_samples) の標本。

    Returns:
        R̂ の値。1.01 未満が目安。
    """
    m, n = chains.shape
    chain_means = chains.mean(axis=1)                    # shape: (m,)
    w = float(chains.var(axis=1, ddof=1).mean())         # 連鎖内分散 W
    b = float(n * chain_means.var(ddof=1))               # 連鎖間分散 B
    var_hat = (n - 1) / n * w + b / n
    return float(np.sqrt(var_hat / w))


N_CHAINS: int = 4
INITS = [0.05, 0.35, 0.65, 0.95]     # わざとバラバラの初期値から出発する

chains = np.empty((N_CHAINS, N_ITER))
rates = []
for j, init in enumerate(INITS):
    chains[j], acc = metropolis(seed=100 + j, init=init)
    rates.append(acc)

kept_chains = chains[:, BURN_IN:]     # shape: (4, 10000)

plt.figure(figsize=(11, 3.8))
for j, init in enumerate(INITS):
    plt.plot(chains[j, :1500], lw=0.7, alpha=0.85, label=f'初期値 {init}')
plt.axvspan(0, BURN_IN, color='C3', alpha=0.12, label='バーンイン')
plt.ylim(0, 1); plt.xlabel('反復回数'); plt.ylabel('θ')
plt.title('4 本の連鎖: 初期値が違っても同じ帯に落ち着けば収束')
plt.legend(ncol=5, fontsize=9); plt.tight_layout(); plt.show()

print(f'各連鎖の受容率  : {[f"{r:.3f}" for r in rates]}')
print(f'各連鎖の事後平均: {[f"{m:.4f}" for m in kept_chains.mean(axis=1)]}')
print(f'厳密な事後平均  : {exact.mean():.4f}')
print(f'R̂ = {gelman_rubin(kept_chains):.4f}   (1.01 未満が目安)')

## 5. JAX 版 — `jit` + `lax.scan` + `vmap`

MCMC は「同じ更新を何万回も繰り返す」処理なので JAX の得意分野です。

- **`lax.scan`**: Python の `for` ループを 1 つの計算グラフに畳み込む
- **`jit`**: コンパイルする
- **`vmap`**: **複数の連鎖を書き直さずに 1 行で**並列化

このあと速度も測りますが、**「JAX なら何でも速い」わけではありません**。
どこで効くのかを実測で確かめます。

> ⚠️ **JAX 特有の注意**
> - `jit` の中で Python の `if theta <= 0` は書けない → `jnp.where` を使う
> - 乱数は `jax.random.PRNGKey` を明示的に `split` する（`np.random` のグローバル状態は使わない）

In [ ]:
import functools

import jax
import jax.numpy as jnp
from jax import lax

A0_J, B0_J, N_J, Y_J = 2.0, 2.0, 20.0, 12.0
PROPOSAL_SD_J: float = 0.08


def log_posterior_jax(theta: jnp.ndarray) -> jnp.ndarray:
    """正規化定数を除いた対数事後密度 (JAX 版)。

    Args:
        theta: 成約率 (スカラー)。

    Returns:
        対数事後密度 (スカラー)。定義域外は -inf。
    """
    inside = (theta > 0) & (theta < 1)
    t = jnp.clip(theta, 1e-12, 1 - 1e-12)     # log(0) 回避 (トレース可能な形で)
    lp = (A0_J + Y_J - 1) * jnp.log(t) + (B0_J + N_J - Y_J - 1) * jnp.log(1 - t)
    return jnp.where(inside, lp, -jnp.inf)    # jit の中では if ではなく where


@functools.partial(jax.jit, static_argnames=("n_iter",))
def metropolis_jax(key: jnp.ndarray, n_iter: int, init: float = 0.5) -> jnp.ndarray:
    """Metropolis 法で対数事後密度からサンプリングする (JAX 版)。

    Args:
        key: jax.random.PRNGKey。
        n_iter: 連鎖の総ステップ数 (static: 変えると再コンパイル)。
        init: 連鎖の初期値。

    Returns:
        標本。shape: (n_iter,)
    """
    def step(carry, k):
        current, current_lp = carry
        k_prop, k_unif = jax.random.split(k)               # 乱数は明示的に分割
        proposal = current + PROPOSAL_SD_J * jax.random.normal(k_prop)
        proposal_lp = log_posterior_jax(proposal)

        accept = jnp.log(jax.random.uniform(k_unif)) < proposal_lp - current_lp
        new = jnp.where(accept, proposal, current)         # 棄却なら現在値を維持
        new_lp = jnp.where(accept, proposal_lp, current_lp)
        return (new, new_lp), new

    init_arr = jnp.float32(init)
    keys = jax.random.split(key, n_iter)                   # shape: (n_iter, 2)
    _, out = lax.scan(step, (init_arr, log_posterior_jax(init_arr)), keys)
    return out                                             # shape: (n_iter,)


chain_j = metropolis_jax(jax.random.PRNGKey(42), N_ITER)
print(f'JAX MCMC 事後平均: {float(chain_j[BURN_IN:].mean()):.4f}   厳密: {exact.mean():.4f}')

# --- vmap で 4 本の連鎖を同時に ---
keys = jax.random.split(jax.random.PRNGKey(0), N_CHAINS)         # shape: (4, 2)
chains_j = jax.vmap(lambda k: metropolis_jax(k, N_ITER))(keys)   # shape: (4, 11000)
chain_means_j = chains_j[:, BURN_IN:].mean(axis=1)               # shape: (4,)
print(f'4 連鎖の事後平均 : {[f"{float(m):.4f}" for m in chain_means_j]}')

# --- 検算: 標準形式 (NumPy) と JAX 形式の結果が一致するか ---
TOLERANCE: float = 0.02
assert abs(float(chain_j[BURN_IN:].mean()) - float(exact.mean())) < TOLERANCE
assert float(chain_means_j.max() - chain_means_j.min()) < TOLERANCE
print('✅ NumPy 版・JAX 版・厳密解がすべて一致')

In [ ]:
def timeit(fn, n_repeat: int = 3) -> float:
    """関数を n_repeat 回実行し、最短の所要時間 (秒) を返す。

    Args:
        fn: 引数なしで呼べる関数。
        n_repeat: 繰り返し回数。最短値を採るのは外乱を除くため。

    Returns:
        最短の所要時間 (秒)。
    """
    times = []
    for _ in range(n_repeat):
        t0 = time.perf_counter()
        fn()
        times.append(time.perf_counter() - t0)
    return min(times)


# --- ① 連鎖 1 本 ---
t_numpy_1 = timeit(lambda: metropolis(seed=7))
metropolis_jax(jax.random.PRNGKey(7), N_ITER).block_until_ready()      # ウォームアップ (初回はコンパイル時間が乗る)
t_jax_1 = timeit(lambda: metropolis_jax(jax.random.PRNGKey(8), N_ITER).block_until_ready())

# --- ② 連鎖をたくさん (NumPy は for で 1 本ずつ、JAX は vmap でまとめて) ---
M_CHAINS: int = 32
vmapped = jax.jit(jax.vmap(lambda k: metropolis_jax(k, N_ITER)))
keys_m = jax.random.split(jax.random.PRNGKey(0), M_CHAINS)             # shape: (32, 2)
vmapped(keys_m).block_until_ready()                                     # ウォームアップ

t_numpy_m = timeit(lambda: [metropolis(seed=200 + j) for j in range(M_CHAINS)], n_repeat=2)
t_jax_m = timeit(lambda: vmapped(keys_m).block_until_ready())

print(f'{"":>8} {"NumPy":>12} {"JAX":>12}   比')
print('-' * 46)
print(f'{"連鎖 1 本":>8} {t_numpy_1*1000:>10.1f} ms {t_jax_1*1000:>10.1f} ms   {t_numpy_1/t_jax_1:.2f} 倍')
print(f'{f"連鎖 {M_CHAINS} 本":>8} {t_numpy_m*1000:>10.1f} ms {t_jax_m*1000:>10.1f} ms   {t_numpy_m/t_jax_m:.2f} 倍')
print()
print('※ 値は環境によって変わります。傾向を見てください。')

### ⚠️ 読み取り — 「JAX なら何でも速い」ではない

実測した傾向（**絶対値は環境で変わります**。手元の CPU での目安）:

| | NumPy | JAX | 比 |
|---|---|---|---|
| 連鎖 1 本 | 約 20 ms | 約 60 ms | **0.3 倍 — JAX のほうが遅い** |
| 連鎖 32 本 | 約 600 ms | 約 110〜170 ms | **4〜6 倍 — JAX が速い** |
| 連鎖 128 本 | 約 2.4 秒 | 約 0.3 秒 | **約 8 倍** |

**1 本だけなら JAX のほうが遅い**。スカラー 1 個ずつの計算では、`jit` / `lax.scan` の間接費のほうが勝ってしまうからです。

連鎖を増やすと `vmap` が効いて逆転します。

> 📌 **JAX の強みは「同じ計算をたくさん」に出ます。**
> - 連鎖を何十本も走らせる（収束診断のため実務では普通）
> - パラメータが 1 個ではなく数百〜数万個ある（ベイズ・ロジスティック回帰、ニューラルネット）
> - GPU / TPU を使う
>
> 逆に「小さいスカラー計算を 1 回」なら素の NumPy が速い。**道具は測って選ぶ**のが正解です。

## 6. 【対話】階層モデルと縮小 — $n$ の小さい店舗を守る

$$
Y_j \mid \theta_j \sim \mathrm{Bin}(n_j, \theta_j), \qquad \theta_j \mid \alpha,\beta \sim \mathrm{Beta}(\alpha, \beta)
$$

店舗ごとの $\theta_j$ は別々だが、$\alpha,\beta$ は**共通**。これが**部分プーリング**です。

結果として、**データの少ない店舗ほど全体平均へ強く引っぱられます**（**縮小 / shrinkage**）。

> ⚠️ ここでは分かりやすさのため、共通分布の中心を全店舗の平均、集中度 $\kappa$ をスライダーで
> 与えた**簡略版**にしています。本来の階層モデルでは $\alpha,\beta$ もデータから推定します。

In [ ]:
# (店舗名, 訪問数 n_j, 成約数 y_j)
STORES = [('A', 2, 2), ('B', 100, 50), ('C', 5, 1), ('D', 30, 21), ('E', 200, 96)]


def show_shrinkage(kappa: float = 20.0) -> None:
    """共通分布の集中度 κ を変えて、店舗ごとの縮小の強さを表示する。

    Args:
        kappa: 共通分布 Beta(α, β) の集中度 α + β。大きいほど全体平均へ強く引く。
    """
    n_all = np.array([n for _, n, _ in STORES])       # shape: (5,)
    y_all = np.array([y for _, _, y in STORES])
    grand_mean = float(y_all.sum() / n_all.sum())     # 全店舗をまとめた成約率

    alpha = kappa * grand_mean                        # 共通分布 Beta(α, β)
    beta_ = kappa * (1 - grand_mean)

    raw = y_all / n_all                                          # 生の観測比率
    shrunk = (alpha + y_all) / (alpha + beta_ + n_all)           # 部分プーリング後

    plt.figure(figsize=(9, 4))
    idx = np.arange(len(STORES))
    plt.scatter(idx, raw, s=90, label='生の観測比率', zorder=3)
    plt.scatter(idx, shrunk, s=90, marker='D', label='縮小後の推定値', zorder=3)
    for i in idx:
        plt.annotate('', xy=(i, shrunk[i]), xytext=(i, raw[i]),
                     arrowprops=dict(arrowstyle='->', color='gray', lw=1.5))
    plt.axhline(grand_mean, color='C3', ls='--', label=f'全体平均 {grand_mean:.3f}')
    plt.xticks(idx, [f'{name}\n(n={n})' for name, n, _ in STORES])
    plt.ylim(0, 1.05); plt.ylabel('成約率')
    plt.title(f'集中度 κ = {kappa:.0f} — n が小さい店舗ほど全体平均へ強く引かれる')
    plt.legend(); plt.tight_layout(); plt.show()

    print(f'{"店舗":>4} {"n":>5} {"生の比率":>9} {"縮小後":>9} {"移動量":>9}')
    print('-' * 42)
    for (name, n, _), r, s in zip(STORES, raw, shrunk):
        print(f'{name:>4} {n:>5} {r:>9.3f} {s:>9.3f} {abs(s-r):>9.3f}')


interact(show_shrinkage,
         kappa=FloatSlider(value=20, min=0.5, max=200, step=0.5, description='集中度 κ'));

### 読み取り

店舗 A は「2 件中 2 件成約 = 100%」ですが、縮小後は全体平均に近づきます。
店舗 E（n=200）はほとんど動きません。

> 💡 実務の直感そのものです。
> **「2 件で 10 割の営業マンより、100 件で 5 割の方が信用できる」**
>
> **判断する質問**: 「この数字、$n$ がいくつのときの割合か？」

## 7. 事後予測チェック — 推定できてもモデルが正しいとは限らない

$$
\theta^{(s)} \sim \pi(\theta \mid x), \qquad \tilde{x}^{(s)} \sim p(\tilde{x} \mid \theta^{(s)})
$$

事後標本ごとに**模擬データ**を作り、実データの特徴（平均・最大値・外れ値数など）を
再現できているか確かめます。「パラメータが推定できた ＝ モデルが正しい」ではありません。

ここでは、**モデルが合っている場合**と**合っていない場合**を並べて比べます。

In [ ]:
N_REP: int = 5_000        # 模擬データセットの本数
ppc_rng = np.random.default_rng(0)

theta_draws = exact.rvs(N_REP, random_state=ppc_rng)     # shape: (5000,) 事後標本

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)

# --- ケース1: モデルが合っている (実データも二項分布から出ている) ---
y_rep = ppc_rng.binomial(N, theta_draws)                 # shape: (5000,) 模擬データ
axes[0].hist(y_rep, bins=np.arange(N + 2) - 0.5, density=True, alpha=0.6)
axes[0].axvline(Y, color='C3', lw=2.5, label=f'実データ y = {Y}')
axes[0].set_title('モデルが合っている場合'); axes[0].set_xlabel('成功数'); axes[0].legend()
p_lo = float((y_rep <= Y).mean())
p_ppc_ok = min(p_lo, 1 - p_lo) * 2

# --- ケース2: モデルが合っていない (実データが極端な値だった場合) ---
Y_WEIRD: int = 19        # 20人中19人成約という「モデルが説明しにくい」観測
axes[1].hist(y_rep, bins=np.arange(N + 2) - 0.5, density=True, alpha=0.6)
axes[1].axvline(Y_WEIRD, color='C3', lw=2.5, label=f'実データ y = {Y_WEIRD}')
axes[1].set_title('モデルが説明できていない場合'); axes[1].set_xlabel('成功数'); axes[1].legend()
p_lo_w = float((y_rep <= Y_WEIRD).mean())
p_ppc_ng = min(p_lo_w, 1 - p_lo_w) * 2

plt.suptitle('事後予測チェック: 模擬データの分布に、実データは収まっているか')
plt.tight_layout(); plt.show()

print(f'ケース1 (y={Y:>2}): 事後予測 p 値相当 = {p_ppc_ok:.4f}  -> 再現できている')
print(f'ケース2 (y={Y_WEIRD:>2}): 事後予測 p 値相当 = {p_ppc_ng:.4f}  -> 0 に近い = モデルを見直す')

## 8. 逐次更新 — 昨日の事後が、今日の事前になる

$$
\pi(\theta \mid x_1) \propto f(x_1 \mid \theta)\pi(\theta), \qquad
\pi(\theta \mid x_1, x_2) \propto f(x_2 \mid \theta)\underbrace{\pi(\theta \mid x_1)}_{\text{昨日の事後}}
$$

Bernoulli–Beta なら、これは単に**足し算を続ける**だけです。
オンライン A/B テストや障害率の継続監視の骨格がこれ。

In [ ]:
# 真の成約率 0.62 のサービスを 10 日間モニタリングする (毎日 20 人)
TRUE_THETA: float = 0.62
DAYS: int = 10
PER_DAY: int = 20

seq_rng = np.random.default_rng(1)
alpha_t, beta_t = A0, B0        # 初日の事前 Beta(2,2)

th = np.linspace(0.001, 0.999, 500)
plt.figure(figsize=(9.5, 4.5))
plt.plot(th, stats.beta.pdf(th, alpha_t, beta_t), color='gray', lw=2, label='初日の事前')

print(f'{"日":>3} {"当日 y/n":>10} {"事後分布":>18} {"事後平均":>9} {"95%信用区間":>20}')
print('-' * 68)
for day in range(1, DAYS + 1):
    y_today: int = int(seq_rng.binomial(PER_DAY, TRUE_THETA))   # その日の成約数 (人)
    alpha_t += y_today                       # 成功側に足す
    beta_t += PER_DAY - y_today              # 失敗側に足す

    post_t = stats.beta(alpha_t, beta_t)
    lo, hi = post_t.ppf([0.025, 0.975])
    print(f'{day:>3} {y_today:>4}/{PER_DAY:<5} '
          f'Beta({alpha_t:>5.0f},{beta_t:>5.0f}) {post_t.mean():>9.4f}'
          f'   [{lo:.3f}, {hi:.3f}]')
    plt.plot(th, post_t.pdf(th), lw=1.6, alpha=0.35 + 0.06 * day)

plt.axvline(TRUE_THETA, color='C3', ls='--', lw=2, label=f'真の値 {TRUE_THETA}')
plt.xlabel('成約率 θ'); plt.ylabel('密度'); plt.xlim(0.2, 1.0)
plt.title('逐次更新: 毎日 20 人ぶんのデータで事後分布が更新されていく')
plt.legend(); plt.tight_layout(); plt.show()

print('\n※ 日を追うごとに山が細くなる = 不確実性が減っている')

### 読み取り

日を追うごとに山が**細く**なり、真の値 0.62 に集中していきます。

> 📌 頻度論の検定は「あらかじめ $n$ を決めて 1 回判定」が原則で、
> 途中で覗き見して逐次に検定すると $\alpha$ 水準が壊れます
> （[`../07_hypothesis_testing.md`](../07_hypothesis_testing.md) の p ハッキング）。
> **ベイズは更新が構造的に定義されている**のが強みです。

## まとめ

- **積分を標本平均に置き換える** — これが今日イチの発想転換
- **MCMC が使うのは 2 地点の「比」だけ**。割り算で正規化定数が消えるので、$\propto$ しか分からなくても困らない
- **棄却されたら現在値をもう一度記録する**。同じ値の連続は仕様（滞在時間が確率になる）
- **対数で比較する**。確率の積はすぐアンダーフローする
- **1 本の連鎖だけで判断しない**。複数連鎖・受容率・$\hat{R}$ で診断する
- **`PROPOSAL_SD` はモデルではなくアルゴリズムのパラメータ**。事後分布そのものは変えない
- **縮小**は「$n$ の小さいグループを守る」仕組み
- **事後予測チェック**は、推定の後に「モデルが妥当か」を問う別の枠

この章の核心:

> **解けない積分を、標本平均に置き換える。**
> **しかも使うのは 2 地点の「比」だけなので、正規化定数は要らない。**

## 確率・統計章、卒業 🎉

確率 → 分布 → 期待値 → ベイズ → 記述統計 → 推定 → 検定 → ベイズ推論 → MCMC と辿ってきました。

→ 次の章: [`../../05_optimization/README.md`](../../05_optimization/README.md)

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次の章 → |
|---|---|---|---|
| [`08_bayesian_inference.ipynb`](08_bayesian_inference.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [`../../05_optimization/README.md`](../../05_optimization/README.md) |